## Recommended Starting Hyperparameters
1. Population Size
* Typical Range: 50–500.
* Recommendation: Start with 100–200.
* Larger populations explore more, but are slower per generation.
* With your large search space, erring toward the higher end (e.g., 200) is reasonable if your fitness function is fast.

2. Number of Generations
* Typical Range: 50–500+.
* Recommendation: 100–300 generations.
* More generations allow more refinement, but diminishing returns after a point.
* You can monitor if the best fitness plateaus early.
3. Selection Method
* Tournament selection or roulette wheel selection are both good.
* Many libraries use tournament selection by default.
4. Crossover Rate
* Typical: 0.7–0.9 (i.e., 70–90% of offspring are produced by crossover).
* Recommendation: 0.8 (80%).
5. Mutation Rate
* Typical: 0.01–0.1 (1–10% mutation rate per gene).
* Recommendation: 0.05 (5% per gene).
* With 9 genes (slots), this means about half your children will have at least one mutation per generation.
6. Elitism
* Keep the best 1–2 individuals from each generation to ensure you don’t lose the best found so far.


In [ ]:
import pandas as pd
from itertools import chain
from functools import reduce

In [2]:
# Hyperparameters
POPULATION_SIZE = 150
NUM_GENERATIONS = 100
MUTATION_RATE = 0.05
EXCLUSIONS = {
    "m_flags" : [3208,3080]
}
LOWER_BOUNDS = {
    # "level" : 190
}
UPPER_BOUNDS = {}

### Load

In [3]:
def load_csv(file_path):
    """Load a CSV file into a DataFrame."""
    df = pd.read_csv(file_path,index_col=0,
                     dtype={k:str for k in ["id", "nameId", "typeId", "set","effectId","characteristic"]})
    df.index = df.index.astype(str)
    return df

In [4]:
items = load_csv("data/datasets/items.csv")
item_sets = load_csv("data/datasets/item_sets.csv")
set_effects = load_csv("data/datasets/set_effects.csv")
effects = load_csv("data/datasets/effects.csv").set_index("id")
item_types = load_csv("data/datasets/item_types.csv").set_index("id").astype(str)

/var/folders/c3/9yq4kl5j7qd715s6mq5tyslm0000gp/T/ipykernel_28181/269983047.py:3: DtypeWarning: Columns (14,15,19,20,21,22,23,24,27,28,29,30,32,33,63,68) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path,index_col=0,


### Filters

In [5]:
for k,v in EXCLUSIONS.items():
    items = items[~items[k].isin(v)]

In [6]:
remove_list = []
for k,v in LOWER_BOUNDS.items():
    remove_list += items[items["type"]!="EffectInstanceDice"][items[k] < v].index.to_list()

for k,v in UPPER_BOUNDS.items():
    remove_list += items[items["type"]!="EffectInstanceDice"][items[k] > v].index.to_list()

items = items.drop(remove_list)

In [7]:
allowed_typeIds = list([str(i) for i in chain(range(1,12), range(16,22))])
weapon_typeIds = list(set(items[items["type"]=="Weapons"]["typeId"].to_list()))
normal_typeIds = [i for i in allowed_typeIds if i not in weapon_typeIds]
extra_typeIds = ["23","151","217"]

In [8]:
def get_item_effects(item):
    if isinstance(item["possibleEffects"], float) or not item["possibleEffects"]:
        # If there are no effects, return an empty Series
        return pd.Series(dtype=float)
    # Get the effects of the item
    item_effects = items.loc[[i for i in item["possibleEffects"].split(",")]]
    # Filter effects to only include those that are in the effects DataFrame
    return item_effects[item_effects["effectId"].isin(effects.index)]

In [9]:
def get_item_contribution(item):
    # Get the effects of the item
    item_effects = get_item_effects(item)
    if item_effects.empty:
        return pd.Series(dtype=float)
    # Merge with effects DataFrame to get characteristics and operators
    item_effects["characteristic"] = effects.loc[item_effects["effectId"]]["characteristic"].values
    item_effects["characteristicOperator"] = effects.loc[item_effects["effectId"]]["characteristicOperator"].apply(lambda x: 1 if x == "+" else -1).values
    # Calculate the maximum value for each effect based on diceSide and diceNum (maximum of the two)
    item_effects["effectValue"] = item_effects["characteristicOperator"] * item_effects[["diceSide", "diceNum"]].max(axis=1)
    # Return a DataFrame with the characteristic and effectValue
    return item_effects[["characteristic", "effectValue"]].set_index("characteristic")["effectValue"].groupby("characteristic").sum()

contributions = {rid: get_item_contribution(items.loc[rid]) for rid in items.index}

In [10]:
def get_set_contribution(item_set,overlap):
    if overlap == 0:
        return pd.Series(dtype=float)
    if item_set.name not in set_effects.index:
        # If the set has no effects, return an empty Series
        return pd.Series(dtype=float)
    # Get the effects of the set based on the overlap
    s_effects = set_effects.loc[item_set.name][str(overlap)]
    if pd.isna(s_effects) or not s_effects:
        # If there are no effects, return an empty Series
        return pd.Series(dtype=float)
    s_effects = s_effects.split(",")
    if "-2" in s_effects:
        # Remove the "-2" from the list of effects
        s_effects.remove("-2")
    s_effects = item_sets.loc[s_effects]
    if s_effects.empty:
        # If there are no effects, return an empty Series
        return pd.Series(dtype=float)
    # Merge with effects DataFrame to get characteristics and operators
    s_effects["characteristic"] = effects.loc[s_effects["effectId"]]["characteristic"].values
    s_effects["characteristicOperator"] = effects.loc[s_effects["effectId"]]["characteristicOperator"].apply(lambda x: 1 if x == "+" else -1).values
    # Calculate the maximum value for each effect based on diceSide and diceNum (maximum of the two)
    s_effects["effectValue"] = s_effects["characteristicOperator"] * s_effects[["diceSide", "diceNum"]].max(axis=1)
    # Return a DataFrame with the characteristic and effectValue
    return s_effects[["characteristic", "effectValue"]].set_index("characteristic")["effectValue"].groupby("characteristic").sum()


In [11]:
set_contributions = {}
for rid,row in item_sets.iterrows():
    if isinstance(row["items"], float) or not row["items"]:
        # If there are no items, return an empty Series
        continue
    set_contributions[rid] = {}
    size = len(row["items"].split(","))
    for i in range(size):
        set_contributions[rid][i+1] = get_set_contribution(row, i+1)
    

## 0. Methods

In [12]:
EXTRA_SLOTS = 6
pools = [items[items["typeId"].isin(weapon_typeIds)].index.to_list()]\
    + [items[items["typeId"].isin([i])].index.to_list() for i in normal_typeIds]
for _ in range(EXTRA_SLOTS):
    pools += [items[items["typeId"].isin(extra_typeIds)].index.to_list()]
NUM_TYPES = len(pools)

In [13]:
import random
def random_chromosome():
    return [random.randint(0, len(slot_pool)-1) for slot_pool in pools]

In [14]:
# Mutation
def mutate(chromosome, mutation_rate=MUTATION_RATE):
    # Create a new chromosome by mutating some slots
    new_chromosome = chromosome[:]
    for i, slot_items in enumerate(pools):
        if random.random() < mutation_rate:
            new_chromosome[i] = random.randint(0, len(slot_items)-1)
    return new_chromosome

In [15]:
# Crossover
def crossover(parent1, parent2):
    # Perform single-point crossover
    point = random.randint(1, NUM_TYPES-2)
    return parent1[:point] + parent2[point:]

In [16]:
# # Sanity check to make sure the function works on all items
# for row,item in items.iterrows():
#     get_item_contribution(item)

In [17]:
# Fitness function
def get_totals(chromosome):
    chromosome_items = [items.loc[pools[idx][item]].name for idx,item in enumerate(chromosome)]
    # Item effects
    lst = [contributions[item] for item in chromosome_items]

    # Set effects
    item_ids = items.loc[chromosome_items]["id"].values
    associated_sets = items.loc[chromosome_items]["set"].dropna().unique()
    for set_rid,set_row in item_sets.loc[associated_sets].iterrows():
        set_items = set_row["items"].split(",")
        overlap = len(set(set_items) & set(item_ids))
        lst.append(set_contributions[set_rid][overlap])
        
    # Totals
    totals = reduce(lambda x, y: x.add(y, fill_value=0), lst).to_dict()
    return totals

def get_damage(weapon, chromosome):
    totals = get_totals(chromosome)
    weapon_effects = get_item_effects(weapon)
    if weapon_effects.empty:
        return 0
    damage_mapper = {
            "98" : "14", # Air,
            "96" : "13", # Water
            "99" : "15", # Fire
            "97" : "10", # Earth
            "100" : "10" # Neutral (by earth)
        }
    power = totals.get("25",0)
    weapon_damage = weapon_effects[weapon_effects["effectId"].isin(damage_mapper)]
    crit_bonus = weapon["criticalHitBonus"] if not pd.isna(weapon["criticalHitBonus"]) else 0
    weapon_crit_chance = weapon["criticalHitProbability"] if not pd.isna(weapon["criticalHitProbability"]) else 0
    crit_chance = weapon_crit_chance + totals.get("18",0)
    crit_chance = crit_chance if crit_chance <=100 else 100
    expected_crit_bonus = crit_bonus * crit_chance/100 # Expected value of distribution
    max_crit_bonus = crit_bonus # Use this for maximum possible
    damage = 0
    for _,row in weapon_damage.iterrows():
        stat = damage_mapper[row["effectId"]]
        weapon_base_damage = row[["diceNum","diceSide"]].max()
        stat_base = totals.get(stat,0)
        damage += (weapon_base_damage + expected_crit_bonus) * (1+(stat_base+power)/100)
    return damage
    

In [18]:
def get_chromosome_weapon(chromosome):
    chromosome_item_ids = [items.loc[pools[idx][item]].name for idx,item in enumerate(chromosome)]
    equipped_items = items.loc[chromosome_item_ids]
    return equipped_items[equipped_items["type"]=="Weapons"].iloc[0]

In [19]:
def fitness(chromosome):
    # Penalizations
    if len(set(chromosome[-EXTRA_SLOTS:])) < EXTRA_SLOTS:
        # Some dofus or trophies are duplicated
        return 0
    # return get_totals(chromosome).get(18, 0)
    weapon = get_chromosome_weapon(chromosome)
    return get_damage(weapon,chromosome)

In [20]:
# Sanity check to make sure the function works on a random chromosome
chromosome = random_chromosome()
assert isinstance(fitness(chromosome), float) or isinstance(fitness(chromosome), int)

## 1. Initialize population

In [21]:
population = [random_chromosome() for _ in range(POPULATION_SIZE)]

## 2. Selection

In [22]:
def select(population, k=10):
    # Select top k chromosomes
    scored = sorted(population, key=fitness, reverse=True)
    return scored[:k]

## 3. Main Loop

In [25]:
items.loc[[pools[idx][item] for idx,item in enumerate(best)]]

,name,type,m_flags,id,nameId,typeId,descriptionId,iconId,level,realWeight,...,effectElement,spellId,effectTriggerDuration,zoneDescr,value,diceNum,diceSide,displayZero,class,set
2861093746004721759,Nueve Colas,Weapons,24712,1511,42479,6,42480.0,6019.0,99.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Espada,NaN
2861093746004729822,Amuleto de Pol Guatson,Items,24712,15701,460411,1,460412.0,1261.0,200.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Amuleto,2861093746001838809
2861093746004729819,Anillo de Pol Guatson,Items,24712,15698,460451,9,808199.0,9296.0,200.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Anillo,2861093746001838809
2861093746004726817,Cinturón de Boreal,Items,24712,12105,32061,10,32062.0,10204.0,185.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cinturón,2861093746001838180
2861093746004733701,Botas de Mil Leguas,Items,57480,20363,779772,11,783433.0,11338.0,200.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Bota,NaN
2861093746004729816,Maskrobio,Items,24712,15695,460429,16,460430.0,16447.0,200.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sombrero,2861093746001838810
2861093746004729818,Velameba,Items,24712,15697,460453,17,460454.0,17330.0,200.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Capa,2861093746001838810
2861093746004727526,Mekrangrejo,Items,24712,13071,97904,18,893522.0,18087.0,20.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Mascota,NaN
2861093746004727753,Dolmanax,Items,16520,13344,241235,23,241236.0,23018.0,100.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Dofus,NaN
2861093746004730368,Lozano mayor,Items,24712,16332,515826,151,515827.0,151204.0,150.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Trofeo,NaN


In [26]:
fitness(best)

np.float64(1273.6053)

In [ ]:
tmp = items.loc[[pools[idx][item] for idx,item in enumerate(best)]]

In [ ]:
item_sets.loc[tmp["set"].dropna()]

,name,type,id,items,nameId,bonusIsSecret,effects,m_flags,effectUid,baseEffectId,...,delay,triggers,effectElement,spellId,effectTriggerDuration,zoneDescr,value,diceNum,diceSide,displayZero
2861093746001838226,Set de bubobubo,ItemSets,276,"14082,14083,14084",297012,0.0,{'Array': [{'values': {'Array': [{'rid': -2}]}...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2861093746001838200,Set de Allister,ItemSets,243,"13123,13124,13125,30060",94699,0.0,{'Array': [{'values': {'Array': [{'rid': -2}]}...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2861093746001838227,Set de nocturlabiúho,ItemSets,277,"14097,14098,14099",297011,0.0,{'Array': [{'values': {'Array': [{'rid': -2}]}...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2861093746001838200,Set de Allister,ItemSets,243,"13123,13124,13125,30060",94699,0.0,{'Array': [{'values': {'Array': [{'rid': -2}]}...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
tmp["set"]

2861093746004729623                    NaN
2861093746004728437    2861093746001838226
2861093746004727557    2861093746001838200
2861093746004728452    2861093746001838227
2861093746004727558    2861093746001838200
2861093746004733698                    NaN
2861093746004733700                    NaN
2861093746004723686                    NaN
2861093746004728192                    NaN
2861093746004727753                    NaN
2861093746001847005                    NaN
2861093746004730294                    NaN
2861093746004730368                    NaN
2861093746004728129                    NaN
Name: set, dtype: object